# Extraction TMDB — Notebook d'exploration

Ce notebook reprend, cellule par cellule, la logique du script `src/extract_api.py`.
L'objectif est de pouvoir **explorer** les données à chaque étape (voir le JSON brut, tester une fonction isolément, visualiser un DataFrame) avant de figer la logique dans le script final.

Pré-requis : un fichier `.env` à la racine du projet contenant `TMDB_API_KEY=votre_cle`.

## 1. Configuration

In [14]:
import os
import json
import time

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"
LANGUE = "fr-FR"

assert API_KEY, "Clé API manquante : vérifiez votre fichier .env"

## 2. Films populaires

On commence par un seul appel, et on regarde la réponse brute avant d'écrire une fonction définitive.

In [25]:
url = f"{BASE_URL}/movie/popular2"
params = {"api_key": API_KEY, "language": LANGUE, "page": 1}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()
donnees_brutes = response.json()

donnees_brutes["results"][0]  # on regarde le premier film pour voir la structure

HTTPError: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/popular2?api_key=76c9319204fae873b7b073c60eaf35b1&language=fr-FR&page=1

In [12]:
# On peut utiliser la fonction `dir()` pour lister toutes les méthodes et attributs disponibles.
dir(response)

['__annotations__',
 '__attrs__',
 '__bool__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__nonzero__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_content',
 '_content_consumed',
 '_next',
 'apparent_encoding',
 'close',
 'connection',
 'content',
 'cookies',
 'elapsed',
 'encoding',
 'headers',
 'history',
 'is_permanent_redirect',
 'is_redirect',
 'iter_content',
 'iter_lines',
 'json',
 'links',
 'next',
 'ok',
 'raise_for_status',
 'raw',
 'reason',
 'request',
 'status_code',
 'text',
 'url']

In [8]:
# Les attributs de la réponse sont accessibles via l'attribut `__attrs__` de l'objet `response`. Cela permet d'obtenir des informations sur les en-têtes, le statut, et d'autres métadonnées de la réponse HTTP.
response.__attrs__

['_content',
 'status_code',
 'headers',
 'url',
 'history',
 'encoding',
 'reason',
 'cookies',
 'elapsed',
 'request']

In [24]:
response.status_code

200

Maintenant qu'on a vu la structure, on formalise l'appel dans une fonction réutilisable.

In [3]:
def get_popular_movies(page=1):
    """Récupère une page de films populaires. Retourne une liste de dicts (ou vide si erreur)."""
    url = f"{BASE_URL}/movie/popular"
    params = {"api_key": API_KEY, "language": LANGUE, "page": page}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur lors de la récupération des films populaires : {erreur}")
        return []

    return response.json().get("results", [])

In [4]:
films_bruts = get_popular_movies(page=1)
print(f"{len(films_bruts)} films récupérés")

# Visualisation rapide sous forme de tableau (pratique pour repérer les champs intéressants)
pd.DataFrame(films_bruts).head()

20 films récupérés


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg,"[878, 28, 12]",969681,Spider-Man : Brand New Day,en,Spider-Man: Brand New Day,"Quatre ans se sont écoulés, Peter, désormais a...",659.5404,/pEEI5mMoyZrbapzWC6vB6UmesDW.jpg,2026-07-29,False,False,7.850,2772
1,False,/7GOW6jod9lLurW5utokAatxg7ql.jpg,"[35, 12, 10751]",1204680,Coyote vs. Acme,en,Coyote vs. Acme,Après des décennies à être réduit en miettes p...,496.9094,/dFwseuSlDUNiwMuElDb3OAOQ8En.jpg,2026-08-20,False,False,7.552,453
2,False,/b9q9VmbXDvJmTziRqkwdEmFdwhr.jpg,"[878, 9648, 53]",1101383,La Fin d'Oak Street,en,The End of Oak Street,Lorsqu’un mystérieux événement cosmique arrach...,467.2595,/sLFjeadRka39hPaLhia8OjGANlc.jpg,2026-08-12,False,False,6.959,1203
3,False,/1CIaRYKf3zg2Xyce1CSfCMg2Vfw.jpg,"[27, 878, 12]",1423191,Resident Evil,en,Resident Evil,"Bryan, un coursier médical, est en train d'eff...",399.1021,/pICoWjcKSet6sA3yzxOkP8ChwYI.jpg,2026-09-16,False,False,7.400,145
4,False,/RMXG8myu1aGlNUsRjtxzmpdMK0.jpg,"[12, 28, 14]",1368337,L'Odyssée,en,The Odyssey,Vingt ans après son départ pour la guerre de T...,367.3657,/kP8jJIkmX0vWEYiqQ9c5zU0dvcn.jpg,2026-07-15,False,False,8.000,3793


## 3. Correspondance des genres

L'endpoint `popular` ne renvoie que des `genre_ids` (des nombres). On récupère la table de correspondance id -> nom.

In [5]:
def get_genre_mapping():
    """Retourne un dict {id_genre: nom_genre}."""
    url = f"{BASE_URL}/genre/movie/list"
    params = {"api_key": API_KEY, "language": LANGUE}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur lors de la récupération des genres : {erreur}")
        return {}

    genres = response.json().get("genres", [])
    return {genre["id"]: genre["name"] for genre in genres}

In [6]:
genres_mapping = get_genre_mapping()
genres_mapping

{28: 'Action',
 12: 'Aventure',
 16: 'Animation',
 35: 'Comédie',
 80: 'Crime',
 99: 'Documentaire',
 18: 'Drame',
 10751: 'Familial',
 14: 'Fantastique',
 36: 'Histoire',
 27: 'Horreur',
 10402: 'Musique',
 9648: 'Mystère',
 10749: 'Romance',
 878: 'Science-Fiction',
 10770: 'Téléfilm',
 53: 'Thriller',
 10752: 'Guerre',
 37: 'Western'}

## 4. Détail d'un film

On teste sur un seul film avant de généraliser (ex. pour enrichir plus tard avec budget/durée).

In [7]:
def get_movie_details(movie_id):
    """Retourne le détail complet d'un film (dict), ou None en cas d'erreur."""
    url = f"{BASE_URL}/movie/{movie_id}"
    params = {"api_key": API_KEY, "language": LANGUE}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as erreur:
        print(f"Erreur lors de la récupération du film {movie_id} : {erreur}")
        return None

    return response.json()

In [8]:
premier_id = films_bruts[0]["id"]
detail = get_movie_details(premier_id)
detail

{'adult': False,
 'backdrop_path': '/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg',
 'belongs_to_collection': {'id': 531241,
  'name': 'Spider-Man (MCU) - Saga',
  'poster_path': '/3BVng0lmJyYIUqm5dxLS2eZ2625.jpg',
  'backdrop_path': '/AvnqpRwlEaYNVL6wzC4RN94EdSd.jpg'},
 'budget': 225000000,
 'genres': [{'id': 878, 'name': 'Science-Fiction'},
  {'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Aventure'}],
 'homepage': 'https://www.sonypictures.fr/film/spider-man-brand-new-day',
 'id': 969681,
 'imdb_id': 'tt22084616',
 'origin_country': ['US'],
 'original_language': 'en',
 'original_title': 'Spider-Man: Brand New Day',
 'overview': "Quatre ans se sont écoulés, Peter, désormais adulte, vit seul, s'est volontairement effacé de la vie et des souvenirs de ses proches. Luttant contre le crime dans un New York qui ne le reconnaît plus, il se consacre entièrement à la protection de la ville - un Spider-Man à plein temps - mais à mesure que les responsabilités s'intensifient, la pression provoque une tr

## 5. Simplification et mise en forme

On ne garde que les champs utiles au projet, et on résout les `genre_ids` en noms de genres.

In [9]:
def extraire_champs_utiles(film, genres_mapping):
    """Simplifie un film TMDB brut en un dict avec les champs utiles au projet."""
    genres_noms = [genres_mapping.get(gid, "Inconnu") for gid in film.get("genre_ids", [])]

    return {
        "id": film.get("id"),
        "titre": film.get("title"),
        "date_sortie": film.get("release_date"),
        "note_moyenne": film.get("vote_average"),
        "nombre_votes": film.get("vote_count"),
        "genres": genres_noms,
        "synopsis": film.get("overview"),
    }

In [10]:
films_simplifies = [extraire_champs_utiles(film, genres_mapping) for film in films_bruts]

df_films = pd.DataFrame(films_simplifies)
df_films.head()

,id,titre,date_sortie,note_moyenne,nombre_votes,genres,synopsis
0,969681,Spider-Man : Brand New Day,2026-07-29,7.850,2772,"[Science-Fiction, Action, Aventure]","Quatre ans se sont écoulés, Peter, désormais a..."
1,1204680,Coyote vs. Acme,2026-08-20,7.552,453,"[Comédie, Aventure, Familial]",Après des décennies à être réduit en miettes p...
2,1101383,La Fin d'Oak Street,2026-08-12,6.959,1203,"[Science-Fiction, Mystère, Thriller]",Lorsqu’un mystérieux événement cosmique arrach...
3,1423191,Resident Evil,2026-09-16,7.400,145,"[Horreur, Science-Fiction, Aventure]","Bryan, un coursier médical, est en train d'eff..."
4,1368337,L'Odyssée,2026-07-15,8.000,3793,"[Aventure, Action, Fantastique]",Vingt ans après son départ pour la guerre de T...


On peut maintenant explorer (tri par note, distribution des genres...) avant de figer quoi que ce soit dans le script.

In [ ]:
df_films.sort_values("note_moyenne", ascending=False).head(10)

## 6. Sauvegarde

Une fois satisfait du résultat, on sauvegarde en JSON — c'est ce même fichier que le script `extract_api.py` produit.

In [11]:
def sauvegarder_json(donnees, chemin_fichier):
    with open(chemin_fichier, "w", encoding="utf-8") as fichier:
        json.dump(donnees, fichier, ensure_ascii=False, indent=2)
    print(f"{len(donnees)} film(s) sauvegardé(s) dans {chemin_fichier}")

sauvegarder_json(films_simplifies, "../data/raw/films_tmdb.json")

20 film(s) sauvegardé(s) dans ../data/raw/films_tmdb.json


---
**Passage au script :** une fois cette exploration validée, la logique retenue est reportée telle quelle dans `src/extract_api.py`, sous forme de fonctions appelées depuis un bloc `if __name__ == "__main__":`. Le notebook reste utile pour explorer de nouvelles pistes (ex. `/movie/{id}/credits` pour le casting) avant de les intégrer au script.